---
title : "Dam Live"
---

In [ ]:
# initialiseer de (toolbox continu inzicht) modules
from pathlib import Path

from toolbox_continu_inzicht.base.config import Config
from toolbox_continu_inzicht.base.data_adapter import DataAdapter

## Inladen van belastingen FEWS HHNK

<details>
<summary>Configuratie</summary>

```yaml
GlobalVariables:
    rootdir: "data_sets/run_damlive"
    moments: [ 0, 24 ]
    calc_time: "2024-11-06 08:00:00"
    aquo_alias:
        H.meting: "WATHTE"
        WATHTE [m][NAP][OW]: "WATHTE"

    LoadsFews:
        host: "https://fews.hhnk.nl"
        port: 443
        region: "fewspiservice"
        version: "1.25"
        parameters: [ "H.meting", "WATHTE [m][NAP][OW]" ]

DataAdapter:
    default_options:
        csv:
            sep: ","
    locaties:
        type: csv
        path: "locations_fews.csv"
    waterstanden:
        type: csv
        path: "hidden_waterstanden_fews.csv"
    waterstanden_xml:
        type: xml_timeseries
        path: "waterstanden_fews.xml"
        parameter_mapping:
            WATHTE: H.meting
        location_mapping:
            MPN-N-24: BP

```

</details>

In [ ]:
config = Config(
    config_path=Path().cwd()
    / "data_sets"
    / "run_damlive"
    / "test_loads_fews_grondwater.yaml"
)
config.lees_config()

data_adapter = DataAdapter(config)

In [ ]:
from toolbox_continu_inzicht.loads.loads_fews.loads_fews import LoadsFews


loads_fews = LoadsFews(data_adapter=data_adapter)

In [ ]:
df_out = loads_fews.run(input="locaties", output="waterstanden_xml")

In [ ]:
data_adapter.input("waterstanden_xml")
file = data_adapter.config.data_adapters["waterstanden_xml"]["abs_path"]
with open(file) as f:
    xml_content = f.read()
# laast bovenste stukje van de xml zien
print(xml_content[:1300])

Voor het geval dat FEWS HHNK niet beschikbaar is, kan met het volgende bestand wel gerekend worden.
In het voorbeeld van FEWS hierboven is het een xml bestand maar elke tabel met meetreeks data kan gebruikt worden.

Bijvoorbeeld de csv:

```yaml
...
waterstanden:
    type: csv
    path: "waterstanden_fews.csv"
...
```

In [ ]:
data_adapter.input("waterstanden_xml")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
df_waterstanden = data_adapter.input("waterstanden_xml")
for station in df_waterstanden["measurement_location_code"].unique():
    df_station = df_waterstanden[
        df_waterstanden["measurement_location_code"] == station
    ].copy()
    df_station.rename(columns={"value": station}, inplace=True)
    df_station.plot(x="date_time", y=station, ax=ax, legend=station, marker=".")

# loads to moments

Een deel van de data vanuit fews komt als kwartier data binnen, om dit naar uren te zetten wordt loads to moments gebruikt

In [ ]:
from toolbox_continu_inzicht.loads import LoadsToMoments

Voor dit voorbeeld doen we nu even 2 tijdstappen

In [ ]:
data_adapter.set_global_variable("moments", [i for i in range(0, 2, 1)])

In [ ]:
loads_to_moments = LoadsToMoments(data_adapter=data_adapter)
loads_to_moments.run(input="waterstanden_xml", output="waterstanden_xml_hourly")

In [ ]:
fig, ax = plt.subplots()
df_waterstanden = loads_to_moments.df_out
for station in df_waterstanden["measurement_location_code"].unique():
    df_station = df_waterstanden[
        df_waterstanden["measurement_location_code"] == station
    ].copy()
    df_station.rename(columns={"value": station}, inplace=True)
    df_station.plot(x="date_time", y=station, ax=ax, legend=station, marker=".")

## Runnen van dam_live

Voor dam live is een licentie van Deltares nodig.

Dit voorbeeld is gemaakt met een versie van damlive versie 26.1.0.7015.


In [ ]:
# initialiseer de (toolbox continu inzicht) modules
from pathlib import Path

from toolbox_continu_inzicht.base.config import Config
from toolbox_continu_inzicht.base.data_adapter import DataAdapter


<details>
<summary> Configuratie</summary>
```yaml
GlobalVariables:
    rootdir: "data_sets/run_damlive"
    moments: [ 0, 24 ]
    calc_time: "2024-11-06 08:00:00"
    UpdateDamLive:
        DAMLIVE_FILE: 'WV2030_Purmer.damx'
        delete_output_folder: False

DataAdapter:
    default_options:
        csv:
            sep: ","
    waterstanden_xml_uur:
        type: xml_timeseries
        path: "waterstanden_fews_hourly.xml"
    parameters_csv:
        type: csv
        index: True
        path: "parameters.csv"
    output_file:
        type: csv
        path: test.csv
    test_live.OutputTimeSeries:
        type: xml_timeseries
        path: live.OutputTimeSeries.xml
``` 
</details>


In [ ]:
config = Config(
    config_path=Path().cwd() / "data_sets" / "run_damlive" / "run_damlive.yaml"
)
config.lees_config()

data_adapter = DataAdapter(config)

De benoemde waterstand metingen tabel moet mee gegeven worden

In [ ]:
data_adapter.input("waterstanden_xml_uur")

En de parameters moeten ook nog worden mee geven, in dit geval rekenen we met bishop

In [ ]:
data_adapter.input("parameters_csv")

Daarnaast moet in de `root_dir` een `{project_naam}.damx` bestand staan wat uitvoer is van een [`Dam stabiliteit`](https://publicwiki.deltares.nl/spaces/DAM/pages/166462239/Aanmaken+DAM+Live+project) berekening, en een map met de stix bestanden: `{project_naam}.geometries2D` waar in het `.damx` bestand naar wordt verwezen. 

In dit voorbeeld is de `project_naam`:  `WV2030_Purmer`. 

Het projectnaam moet meegeven worden in de opties, 

In [ ]:
from toolbox_continu_inzicht.dam_live import UpdateDamLive

update_dam_live = UpdateDamLive(data_adapter=data_adapter)

In [ ]:
# om meer informatie terug te krijgen van DAMLIVE, kan de logging level op INFO worden gezet.
update_dam_live.data_adapter.set_global_variable("logging", {"level": "INFO"})
update_dam_live.data_adapter.init_logging(re_initialize=True)

Dit duurt ca 2 min per tijdstap: 

In [ ]:
update_dam_live.run(
    input=["waterstanden_xml_uur", "parameters_csv"], output="output_file"
)

In [ ]:
update_dam_live.data_adapter.input("test_live.OutputTimeSeries")